# Solar Filament Segmentation - v2 (higher resolution, crop training, PQ-tuned post-processing)
Changes vs v1: trains on random **crops from 1024px images** (4x finer detail at the same CPU cost), predicts on full 1024px images,
adds a local-contrast input channel, uses all annotators for validation PQ, fills holes, and searches a wider post-processing grid.
Run all cells. Output: `submission.csv`. Edit only cell 2 if needed.

%pip install -q segmentation-models-pytorch pycocotools opencv-python-headless pandas numpy tqdm

In [ ]:
\# 2) CONFIG
import os, json, glob, time, zipfile, itertools, csv, math
from collections import defaultdict
import numpy as np, cv2, pandas as pd, torch, torch.nn.functional as F
from tqdm.auto import tqdm
from pycocotools import mask as mu
import segmentation_models_pytorch as smp

ZIP_PATH = r"C:\Users\shrut\Desktop\final\filament-segmentation-2026.zip"
WORK_DIR = "./work"
TIME_BUDGET_MIN = None   # training minutes. None = auto (CPU 30, GPU 15). More time = better score.
SEED = 0

device = "cuda" if torch.cuda.is_available() else "cpu"
RES, EVAL_RES = 1024, 512          # working resolution / resolution used to tune post-processing
if device == "cuda":
    ENCODER, CROP, BS, BUDGET, TTA = "resnet34", 512, 8, (TIME_BUDGET_MIN or 15)*60, True
else:
    ENCODER, CROP, BS, BUDGET, TTA = "resnet18", 256, 8, (TIME_BUDGET_MIN or 30)*60, False
    torch.set_num_threads(os.cpu_count())
use_amp = device == "cuda"
print("device:", device, "| encoder", ENCODER, "| crop", CROP, "| train min:", BUDGET/60)
rng = np.random.default_rng(SEED); torch.manual_seed(SEED)

c:\Users\shrut\Desktop\final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cpu | encoder resnet18 | crop 256 | train min: 30.0


In [2]:
# 3) unzip + locate files
data_dir = os.path.join(WORK_DIR, "data")
if not os.path.isdir(data_dir):
    os.makedirs(data_dir, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(data_dir)
ann_path = glob.glob(os.path.join(data_dir, "**", "*Annotations*train*.json"), recursive=True)[0]
train_dir = [d for d in glob.glob(os.path.join(data_dir, "**", "train_images"), recursive=True) if os.path.isdir(d)][0]
test_dir  = [d for d in glob.glob(os.path.join(data_dir, "**", "test_images"),  recursive=True) if os.path.isdir(d)][0]
get_stem = lambda p: os.path.splitext(os.path.basename(p))[0][-16:]
train_files = {get_stem(p): p for p in glob.glob(os.path.join(train_dir, "*"))}
test_files  = {get_stem(p): p for p in sorted(glob.glob(os.path.join(test_dir, "*")))}
print(len(train_files), "train files,", len(test_files), "test files")

707 train files, 180 test files


In [3]:
# 4) dataset. Inputs = 2 channels: raw intensity + local contrast (image minus large-scale blur, removes limb darkening)
def make_inputs(gray, res=RES):
    x = cv2.resize(gray, (res, res), interpolation=cv2.INTER_AREA)
    blur = cv2.GaussianBlur(x.astype(np.float32), (0, 0), 30)
    d = np.clip((x.astype(np.float32) - blur) * 2 + 128, 0, 255).astype(np.uint8)
    return x, d

def to_tensor(x, d):   # uint8 arrays (..., H, W) -> float tensor (N, 2, H, W)
    x = torch.from_numpy(np.ascontiguousarray(x)).float(); d = torch.from_numpy(np.ascontiguousarray(d)).float()
    return torch.stack([(x/255 - 0.45)/0.25, (d - 128)/2/40], 1)

coco = json.load(open(ann_path))
anns_by_img = defaultdict(list)
for a in coco["annotations"]: anns_by_img[a["image_id"]].append(a)
groups = defaultdict(list)
for im in coco["images"]:
    s = get_stem(im["file_name"])
    if s in train_files: groups[s].append(im)
for s in groups: groups[s].sort(key=lambda d: str(d["id"]))
stems = sorted(groups); print("unique train images:", len(stems))
perm = rng.permutation(len(stems)); n_val = max(8, int(0.15*len(stems)))
val_stems = [stems[i] for i in perm[:n_val]]; tr_stems = [stems[i] for i in perm[n_val:]]

def poly_masks(img_info, res):   # one binary mask per filament, at resolution res
    sc = res / img_info["width"]; inst = []
    for a in anns_by_img[img_info["id"]]:
        m = np.zeros((res, res), np.uint8)
        for seg in a["segmentation"]:
            cv2.fillPoly(m, [np.round(np.array(seg, dtype=np.float64).reshape(-1, 2) * sc).astype(np.int32)], 1)
        inst.append(m)
    return inst

def union_soft(s, res):   # mean over annotators of their union mask, stored as uint8 0..255
    ms = []
    for im in groups[s]:
        p = poly_masks(im, res)
        ms.append(np.clip(np.sum(p, 0), 0, 1) if p else np.zeros((res, res), np.uint8))
    return np.round(np.mean(ms, 0) * 255).astype(np.uint8)

def read_gray(path): return cv2.imread(path, cv2.IMREAD_GRAYSCALE)

Xtr, Dtr, Ytr = [], [], []
for s in tqdm(tr_stems):
    x, d = make_inputs(read_gray(train_files[s])); Xtr.append(x); Dtr.append(d); Ytr.append(union_soft(s, RES))
Xtr, Dtr, Ytr = np.stack(Xtr), np.stack(Dtr), np.stack(Ytr)
fg_idx = [np.flatnonzero(y.ravel() > 127) for y in Ytr]   # foreground pixels, used to centre crops on filaments

Xva, Dva, GTva = [], [], []
for s in tqdm(val_stems):
    x, d = make_inputs(read_gray(train_files[s])); Xva.append(x); Dva.append(d)
    GTva.append([[m.astype(bool) for m in poly_masks(im, EVAL_RES)] for im in groups[s]])   # one instance list per annotator
Xva, Dva = np.stack(Xva), np.stack(Dva)
print(Xtr.shape, Xva.shape)

unique train images: 707


100%|██████████| 106/106 [00:06<00:00, 15.60it/s]

(601, 1024, 1024) (106, 1024, 1024)


In [4]:
# 5) model + training on random crops (time-based cosine LR, stops when budget is used)
try:
    model = smp.Unet(ENCODER, encoder_weights="imagenet", in_channels=2, classes=1)
    print("loaded ImageNet-pretrained encoder")
except Exception as e:
    print("!! pretrained weights unavailable (", type(e).__name__, ") -> from scratch (needs more training time)")
    model = smp.Unet(ENCODER, encoder_weights=None, in_channels=2, classes=1)
model.to(device)

def get_batch():
    xs, ds, ys = [], [], []
    for _ in range(BS):
        i = int(rng.integers(len(Xtr)))
        if rng.random() < 0.8 and len(fg_idx[i]):
            p = int(fg_idx[i][rng.integers(len(fg_idx[i]))]); cy, cx = divmod(p, RES)
            y0 = int(np.clip(cy - CROP//2 + rng.integers(-CROP//4, CROP//4 + 1), 0, RES - CROP))
            x0 = int(np.clip(cx - CROP//2 + rng.integers(-CROP//4, CROP//4 + 1), 0, RES - CROP))
        else:
            y0, x0 = int(rng.integers(0, RES - CROP + 1)), int(rng.integers(0, RES - CROP + 1))
        sl = (slice(y0, y0 + CROP), slice(x0, x0 + CROP))
        a, b, c = Xtr[i][sl], Dtr[i][sl], Ytr[i][sl]
        if rng.random() < .5: a, b, c = a[:, ::-1], b[:, ::-1], c[:, ::-1]
        if rng.random() < .5: a, b, c = a[::-1], b[::-1], c[::-1]
        k = int(rng.integers(4)); a, b, c = np.rot90(a, k), np.rot90(b, k), np.rot90(c, k)
        xs.append(a); ds.append(b); ys.append(c)
    t = to_tensor(np.stack(xs), np.stack(ds))
    t[:, 0] = t[:, 0] * float(rng.uniform(0.9, 1.1)) + float(rng.uniform(-0.1, 0.1))   # brightness/contrast jitter
    return t.to(device), (torch.from_numpy(np.stack(ys)).float() / 255).unsqueeze(1).to(device)

def loss_fn(lg, y):
    bce = F.binary_cross_entropy_with_logits(lg, y)
    p = torch.sigmoid(lg); inter = (p*y).sum((1,2,3))
    dice = 1 - (2*inter + 1) / (p.sum((1,2,3)) + y.sum((1,2,3)) + 1)
    return bce + dice.mean()

BASE_LR = 5e-4
opt = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
t0 = time.time(); it = 0; run = []
model.train()
while True:
    frac = (time.time() - t0) / BUDGET
    if frac >= 1: break
    for g in opt.param_groups: g["lr"] = BASE_LR * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * frac)))
    x, y = get_batch()
    with torch.autocast(device_type=device, enabled=use_amp):
        lg = model(x)
    loss = loss_fn(lg.float(), y)
    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    run.append(loss.item()); it += 1
    if it % 50 == 0:
        print(f"iter {it} | loss {np.mean(run[-50:]):.4f} | {(time.time()-t0)/60:.1f}/{BUDGET/60:.0f} min")
torch.save(model.state_dict(), os.path.join(WORK_DIR, "model_v2.pt")); print("trained", it, "iterations")

loaded ImageNet-pretrained encoder


C:\Users\shrut\AppData\Local\Temp\ipykernel_34640\3954650934.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


iter 50 | loss 1.3414 | 1.6/30 min
iter 100 | loss 0.9978 | 3.3/30 min
iter 150 | loss 0.7841 | 5.0/30 min
iter 200 | loss 0.6259 | 6.9/30 min
iter 250 | loss 0.5745 | 8.9/30 min
iter 300 | loss 0.5050 | 10.9/30 min
iter 350 | loss 0.4842 | 12.9/30 min
iter 400 | loss 0.4886 | 14.8/30 min
iter 450 | loss 0.4625 | 16.8/30 min
iter 500 | loss 0.4520 | 18.8/30 min
iter 550 | loss 0.4552 | 20.7/30 min
iter 600 | loss 0.4433 | 22.6/30 min
iter 650 | loss 0.4534 | 23.9/30 min
iter 700 | loss 0.4353 | 25.4/30 min
iter 750 | loss 0.4341 | 27.4/30 min
iter 800 | loss 0.4369 | 29.4/30 min
trained 816 iterations


In [5]:
# 6) full-image inference (1024px) + validation probabilities
@torch.no_grad()
def predict_probs(X, D, bs=2):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        x = to_tensor(X[i:i+bs], D[i:i+bs]).to(device)
        with torch.autocast(device_type=device, enabled=use_amp):
            p = torch.sigmoid(model(x).float())
            if TTA: p = (p + torch.sigmoid(model(x.flip(-1)).float()).flip(-1)) / 2
        out.append(p.squeeze(1).cpu().numpy())
    return np.concatenate(out)

Pva_full = predict_probs(Xva, Dva)
Pva = np.stack([cv2.resize(p, (EVAL_RES, EVAL_RES), interpolation=cv2.INTER_AREA) for p in Pva_full])

In [6]:
# 7) post-processing (instances) + PQ over ALL annotators + parameter search
def instances(prob, thr, k, min_area, s=1.0):
    m = (prob > thr).astype(np.uint8)
    kk = int(round(k * s))
    if kk > 1: m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kk, kk)))
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)   # fill holes
    m = np.zeros_like(m); cv2.drawContours(m, cnts, -1, 1, thickness=-1)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    return [lab == i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= min_area * s * s]

def pq_counts(gts, preds):
    if not gts or not preds: return 0.0, 0, len(preds), len(gts)
    G = np.stack([g.ravel() for g in gts]).astype(np.float32); P = np.stack([p.ravel() for p in preds]).astype(np.float32)
    inter = G @ P.T; iou = inter / (G.sum(1)[:, None] + P.sum(1)[None] - inter + 1e-9)
    match = iou > 0.5; tp = int(match.sum())
    return float(iou[match].sum()), tp, len(preds) - tp, len(gts) - tp

def eval_pq(thr, k, min_area):
    s_iou = tp = fp = fn = 0
    for prob, annots in zip(Pva, GTva):
        preds = instances(prob, thr, k, min_area)
        for gt in annots:
            a, b, c, d = pq_counts(gt, preds); s_iou += a; tp += b; fp += c; fn += d
    return s_iou / max(tp + .5*fp + .5*fn, 1e-9)

best = (-1, None)
grid = list(itertools.product([0.25, 0.3, 0.4, 0.5, 0.6], [0, 5, 9, 15, 25], [15, 40, 100, 250]))
for thr, k, ma in tqdm(grid):
    pq = eval_pq(thr, k, ma)
    if pq > best[0]: best = (pq, (thr, k, ma))
print("best val PQ: %.4f with (thr, close_k, min_area) = %s  (units: pixels at %dpx)" % (best[0], best[1], EVAL_RES))
THR, K, MINA = best[1]

100%|██████████| 100/100 [04:06<00:00,  2.47s/it]

best val PQ: 0.3360 with (thr, close_k, min_area) = (0.3, 0, 40)  (units: pixels at 512px)


In [7]:
# 8) predict test set -> submission.csv
rows = []; n_img_with = 0
for stem, path in tqdm(test_files.items()):
    g = read_gray(path); H, W = g.shape
    x, d = make_inputs(g)
    prob = cv2.resize(predict_probs(x[None], d[None])[0], (W, H), interpolation=cv2.INTER_LINEAR)
    s = W / EVAL_RES
    insts = instances(prob, THR, K, MINA, s=s)
    if not insts:   # fallback: biggest blob at a lower threshold
        insts = sorted(instances(prob, 0.2, 0, MINA / 2, s=s), key=lambda m: -m.sum())[:1]
    n_img_with += bool(insts)
    for i, m in enumerate(insts, 1):
        r = mu.encode(np.asfortranarray(m.astype(np.uint8)))
        rows.append((f"{stem}_{i}", r["counts"].decode()))
sub = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
sub.to_csv("submission.csv", index=False, quoting=csv.QUOTE_NONE)
print(len(sub), "filaments over", n_img_with, "/", len(test_files), "images -> submission.csv")
sub.head()

100%|██████████| 180/180 [05:37<00:00,  1.87s/it]

1938 filaments over 180 / 180 images -> submission.csv


,filament_id,segmentation_rle
0,20110120105534Ch_1,n_Vi25io14N1O2N100O2O2M1kPN_Ojn1b0SQN@mn1`0QQN...
1,20110120105534Ch_2,kjPo18go12M3O0O010O010O001O00000000O1L50O00100...
2,20110120105534Ch_3,QfY_19eo14M4L2M4N1O1O0010000O1O1O1O001O0O10O01...
3,20110130110334Ch_1,\Xnm18`o1:N1N1O2O0O2O1N3N2N2N2M2O1G9N3M2O1O1O2...
4,20110130110334Ch_2,b[no17eo17J5K6F8M4M1N3M2G9N1YOgNgRNZ1Ym1lNaRNT...
